# the plan

3 types of clustering:

- heirarchical clustering
- k-means and other clustering methods in UMAP/ISOMAP space
- MDL (naive + greedy)

then:

- 3d vis of these clusters
- stats on how the clusters match up

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from muutils.dbg import dbg_auto, dbg_tensor

from spd.analysis.embedding import get_comp_dist_mat, plot_embedding_result, sweep_embedding_param
from spd.analysis.grouping import (
    CoactivationResults,
    coactivation_hierarchical_clustering,
    get_coactivations,
)
from spd.data_utils import SparseFeatureDataset
from spd.experiments.resid_mlp.resid_mlp_dataset import ResidualMLPDataset
from spd.utils import get_device

DEVICE = get_device()
torch.set_grad_enabled(False)
print(f"Using device: {DEVICE = }")

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

# TMS + identity

In [ ]:
coactivations_tms: CoactivationResults = get_coactivations(
    model_path=Path("../data/tms-decomp/model_40000.pth"),
    dataset_cls=SparseFeatureDataset,
    dataset_kwargs=dict(
        value_range=(0.0, 1.0),
        synced_inputs=None,
    ),
    coactivations_kwargs=dict(
        module_groups=[["linear1", "linear2"]],
    ),
    device=DEVICE,
)

In [ ]:
dbg_auto(coactivations_tms);

In [ ]:
hclust_res = coactivation_hierarchical_clustering(
    coactivations_tms["group_0"],
)

dbg_auto(hclust_res);

In [ ]:
tms_dist = get_comp_dist_mat(coactivations_tms["group_0"])
dbg_tensor(tms_dist)

for method in ["umap", "isomap"]:
    plot_embedding_result(
        sweep_embedding_param(
            dist=tms_dist,
            method=method,
            param_name="n_neighbors",
            param_values=[2, 3, 4, 8, 16, 32, 64, 128, 256],
            n_components=2,
        )
    )

In [ ]:
from spd.analysis.embedding import sweep_embeddings_and_clusters

sweep_embeddings_and_clusters(tms_dist)

# 1 layer residual MLP

In [ ]:
coactivations_mlp: CoactivationResults = get_coactivations(
    model_path=Path("../data/mlp-decomp/model_30000.pth"),
    dataset_cls=ResidualMLPDataset,
    dataset_kwargs=dict(
        calc_labels=False,  # Our labels will be the output of the target model
        label_type=None,
        act_fn_name=None,
        label_fn_seed=None,
        label_coeffs=None,
        synced_inputs=None,
    ),
    coactivations_kwargs=dict(
        module_groups=[["layers.0.mlp_in", "layers.0.mlp_out"]],
    ),
    device=DEVICE,
)
coactivation_hierarchical_clustering(
    results=coactivations_mlp,
    title="MLP Coactivations",
);

# 3 layer residual MLP

In [ ]:
coactivations_mlp3: CoactivationResults = get_coactivations(
    model_path=Path("../data/mlp3-decomp/model_120000.pth"),
    dataset_cls=ResidualMLPDataset,
    dataset_kwargs=dict(
        calc_labels=False,  # Our labels will be the output of the target model
        label_type=None,
        act_fn_name=None,
        label_fn_seed=None,
        label_coeffs=None,
        synced_inputs=None,
    ),
    coactivations_kwargs=dict(
        module_groups=[
            [
                "layers.0.mlp_in",
                "layers.0.mlp_out",
                "layers.1.mlp_in",
                "layers.1.mlp_out",
                "layers.2.mlp_in",
                "layers.2.mlp_out",
            ]
        ],
    ),
    device=DEVICE,
)

_, mlp3_clusters, _, mlp3_alive_mask = coactivation_hierarchical_clustering(
    coactivations_mlp3,
    group_key="group_0",
    linkage_method="average",
    threshold=0.8,
    # linkage_method="complete",
    # threshold=0.85,
    title="3-layer Residual MLP Hierarchical Clustering",
    min_alive_counts=500,
    figsize=(15, 6),
)

In [ ]:
submodules = coactivations_mlp3["group_0"]["labels"][mlp3_alive_mask]
# Get top 10 clusters by size
cluster_sizes = pd.Series(mlp3_clusters).value_counts()

for cluster_id in cluster_sizes.index[0::10]:
    cluster_mask = mlp3_clusters == cluster_id
    submodule_counts = pd.Series(submodules[cluster_mask]).value_counts()

    print(f"\nCluster {cluster_id} ({cluster_sizes[cluster_id]} elements):")
    for submodule, count in submodule_counts.items():
        print(f"  {submodule}: {count}")